# Lab 39: Convolutional Neural Networks (CNNs) for Image Classification

**Duration:** 3 hours. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
The classifier of Lab 9 and the detectors of Labs 10 to 14 are all built from **convolutional neural networks (CNNs)**. A CNN is a neural network made for pictures. Instead of looking at every pixel separately, it slides a small **filter** over the picture and looks for patterns such as edges and curves, in the same way in every place. In this lab you train a small CNN on the GPU of the Jetson to read handwritten digits, look at the filters and the feature maps it learned, and compare it with a plain network (an MLP) that treats the picture as a list of numbers.

**By the end of this lab you can:**
1. Explain a convolution (filter, stride, padding) and pooling, and what a feature map is.
2. Build a small CNN in PyTorch and train it on the GPU.
3. Compare the CNN with an MLP on accuracy, number of weights and robustness.
4. Look at the filters and the feature maps and say what the network learned to see.
5. Read a confusion matrix of ten classes and find the digits that are mixed up.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Training from scratch* on a small dataset (1797 pictures of 8 x 8 pixels that come with scikit-learn, nothing is downloaded). The same ideas apply to the big CNNs of the YOLO labs.

## 2. Background

**Why not just use a normal network?** A picture of 8 x 8 pixels has 64 numbers. A picture of 640 x 640 pixels in colour has 1.2 million numbers. A plain network that connects every pixel to every neuron would need billions of weights and would not know that two pixels next to each other belong together, or that a digit moved one pixel to the right is the same digit.

**The convolution.** A **filter** (also called a kernel) is a tiny grid of weights, for example 3 x 3. It slides over the picture. At every place it multiplies its weights with the pixels under it and adds them: one number. The result of one filter over the whole picture is a new picture, the **feature map**. A filter learns to react to one pattern (a vertical edge, a curve). The **same** weights are used everywhere, so a CNN needs few weights and recognises a pattern wherever it is.

| Word | Meaning |
|---|---|
| Filter / kernel | The small grid of weights (3 x 3) |
| Channels | Pictures stacked: 1 for grey, 3 for colour; after a layer, one channel for each filter |
| Padding | Zeros around the picture, so the feature map keeps the size |
| Stride | How many pixels the filter jumps (1 = every pixel) |
| Feature map | The output picture of one filter |
| Pooling | Keeps the biggest number of each 2 x 2 block: the picture becomes half as big and a small shift matters less |

**The structure.** Convolution, activation (ReLU), convolution, activation, pooling, and then a normal layer that decides the class. Early layers find edges, later layers find parts, the last layer decides.

**Counting weights.** A convolution with `in` channels, `out` filters and a 3 x 3 kernel has `out x (in x 3 x 3 + 1)` weights. A linear layer from `a` to `b` numbers has `a x b + b`.

**Training on the GPU.** A CNN does thousands of the same small calculations at once. This is what the GPU is good at (Lab 5). You move the model and the data to the GPU with `.to("cuda")`.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(39)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 3
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU")

---
# Part B: The Pictures (25 min)

### Step 1: Look at the Digits
`load_digits` has 1797 handwritten digits, each a grey picture of 8 x 8 pixels (values 0 to 16). The cell shows the first 10 and splits the data 60 % train, 20 % validation, 20 % test.

In [ ]:
digits = load_digits()
images = (digits.images / 16.0).astype(np.float32)            # numbers from 0 to 1
labels = digits.target

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, image, label in zip(axes, images[:10], labels[:10]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

X_rest, X_test, y_rest, y_test = train_test_split(images, labels, test_size=0.2, random_state=SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=SEED, stratify=y_rest)
print("train", len(X_train), "| validation", len(X_val), "| test", len(X_test), "| picture shape", images[0].shape)

def to_tensor(x, y):
    return torch.tensor(x).unsqueeze(1).to(device), torch.tensor(y).to(device)      # (N, 1, 8, 8): one channel


Xt, yt = to_tensor(X_train, y_train)
Xv, yv = to_tensor(X_val, y_val)
Xs, ys = to_tensor(X_test, y_test)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many pictures there are in each set:
- Which digit looks the hardest to read to you:
- What is the shape (N, 1, 8, 8): what are the four numbers:

---
# Part C: One Convolution by Hand (30 min)

### Step 1: A Filter That Finds Vertical Edges
A 3 x 3 filter with `-1, 0, 1` in every row reacts where the picture goes from dark to bright from left to right. The cell slides this filter over one digit with two loops (the slow way) and over the whole picture with PyTorch, and shows the feature map.

In [ ]:
image = images[1]                                    # a handwritten 1
kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)

padded = np.pad(image, 1)
slow = np.zeros_like(image)
for row in range(8):
    for col in range(8):
        slow[row, col] = (padded[row:row + 3, col:col + 3] * kernel).sum()

fast = nn.functional.conv2d(torch.tensor(image)[None, None], torch.tensor(kernel)[None, None], padding=1)[0, 0].numpy()
print("Both ways give the same feature map:", np.allclose(slow, fast, atol=1e-5))

fig, axes = plt.subplots(1, 2, figsize=(5, 2.4))
axes[0].imshow(image, cmap="gray_r")
axes[0].set_title("picture")
axes[1].imshow(slow, cmap="coolwarm")
axes[1].set_title("vertical-edge filter")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Do the two ways agree:
- Where in the feature map are the strongest red and blue values, compared with the picture:
- How many multiplications does the slow way make for one 8 x 8 picture with one 3 x 3 filter:

---
# Part D: Build and Train the CNN (50 min)

### Step 1: Two Models
The **MLP** flattens the picture (64 numbers) and uses two normal layers. The **CNN** has two convolutions, pooling, and one normal layer. The cell counts the weights of each. Note that the CNN sees the picture as a picture.

In [ ]:
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)).to(device)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),            # 8 filters: (N, 8, 8, 8)
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),           # 16 filters: (N, 16, 8, 8)
    nn.MaxPool2d(2),                                     # half the size: (N, 16, 4, 4)
    nn.Flatten(),                                        # 256 numbers
    nn.Linear(16 * 4 * 4, 10),                           # ten classes
).to(device)


def count(model):
    return sum(p.numel() for p in model.parameters())


print("MLP weights:", count(mlp))
print("CNN weights:", count(cnn), "(first convolution:", 8 * (1 * 9 + 1), "| second:", 16 * (8 * 9 + 1), "| last layer:", 256 * 10 + 10, ")")
print("Shape after each layer of the CNN:")
x = Xt[:1]
for layer in cnn:
    x = layer(x)
    print(f"  {layer.__class__.__name__:<10} -> {tuple(x.shape)}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The number of weights of the MLP and of the CNN:
- Which is smaller:
- The shape after the pooling layer, and why it is half as big:

### Step 2: Train Both
The same loop trains both models: Adam, cross entropy, batches of 64, 30 epochs. The cell prints the time, because the CNN runs on the GPU.

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()


def train(model, epochs=30, lr=2e-3, batch=64):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(Xt), device=device)
        for start in range(0, len(Xt), batch):
            idx = order[start:start + batch]
            optimiser.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()
            optimiser.step()
        history["loss"].append(loss.item())
        history["val_acc"].append(accuracy(model, Xv, yv))
    return history


results = {}
for name, model in (("MLP", mlp), ("CNN", cnn)):
    torch.manual_seed(SEED)
    for layer in model.modules():
        if hasattr(layer, "reset_parameters"):
            layer.reset_parameters()
    started = time.time()
    results[name] = train(model)
    if device == "cuda":
        torch.cuda.synchronize()
    print(f"{name}: trained in {time.time() - started:4.1f} s | validation accuracy {results[name]['val_acc'][-1] * 100:.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Validation accuracy of the MLP and of the CNN:
- How long each took:
- Which model has fewer weights and a higher accuracy, and what does it tell you:

### Step 3: The Learning Curves
The validation accuracy of both models over the epochs.

In [ ]:
plt.figure(figsize=(5, 3))
for name, history in results.items():
    plt.plot(history["val_acc"], label=name)
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.ylim(0.7, 1.0)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# Part E: Look Inside the CNN (40 min)

### Step 1: The Filters
The first layer has 8 filters of 3 x 3. They are the weights the network learned. Many look like edge and corner detectors, as in the eye of an animal.

In [ ]:
filters = cnn[0].weight.detach().cpu().numpy()[:, 0]
fig, axes = plt.subplots(1, 8, figsize=(10, 1.5))
for ax, f in zip(axes, filters):
    ax.imshow(f, cmap="coolwarm")
    ax.axis("off")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Describe two of the filters (which pattern do they react to):
- Why are the filters not exactly like the one you wrote by hand:

### Step 2: The Feature Maps of One Digit
The cell sends one digit through the first layer and draws the 8 feature maps. Each map shows where one filter found its pattern.

In [ ]:
sample = Xs[3:4]
with torch.no_grad():
    maps = torch.relu(cnn[0](sample))[0].cpu().numpy()
fig, axes = plt.subplots(1, 9, figsize=(11, 1.6))
axes[0].imshow(sample[0, 0].cpu().numpy(), cmap="gray_r")
axes[0].set_title("digit " + str(int(ys[3])))
for ax, m in zip(axes[1:], maps):
    ax.imshow(m, cmap="viridis")
for ax in axes:
    ax.axis("off")
plt.show()

### Step 3: Shift the Pictures by One Pixel
A digit moved one pixel to the right is still the same digit. The MLP has never been told this. The CNN, which uses the same filter everywhere, is more robust. The cell shifts all test pictures and measures both models.

In [ ]:
shifted = torch.roll(Xs, shifts=1, dims=3)             # move every picture one pixel to the right
for name, model in (("MLP", mlp), ("CNN", cnn)):
    print(f"{name}: test accuracy {accuracy(model, Xs, ys) * 100:.1f} %   after a shift of 1 pixel {accuracy(model, shifted, ys) * 100:.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The accuracy of both models before and after the shift:
- Which model lost more, and why:
- Where does this matter in a real camera application:

### Step 4: Where Does It Go Wrong?
The confusion matrix of the CNN on the test set.

In [ ]:
cnn.eval()
with torch.no_grad():
    predicted = cnn(Xs).argmax(1).cpu().numpy()
ConfusionMatrixDisplay(confusion_matrix(y_test, predicted)).plot(cmap="Blues", colorbar=False)
plt.show()
wrong = np.where(predicted != y_test)[0]
print("Wrong:", len(wrong), "of", len(y_test), "| pairs:", [(int(y_test[i]), int(predicted[i])) for i in wrong[:8]])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many test pictures were wrong:
- Which two digits were mixed up and why does that make sense:

---
# Part F: Compare with the Classifier of Lab 9 (15 min, optional)
The classifier `yolo26n-cls` of Lab 9 is also a CNN. Count its weights. It has the same building blocks as your small network, only many more of them.

In [ ]:
import os
try:
    from ultralytics import YOLO
    path = os.path.join(os.environ["LAB_HOME"], "models", "yolo26n-cls.pt")
    model = YOLO(path if os.path.exists(path) else "yolo26n-cls.pt")
    weights = sum(p.numel() for p in model.model.parameters())
    convolutions = sum(1 for m in model.model.modules() if isinstance(m, nn.Conv2d))
    print(f"yolo26n-cls: {weights:,} weights, {convolutions} convolution layers   (your CNN: {count(cnn):,} weights, 2 convolution layers)")
except Exception as error:
    print("Skipped (the model of Lab 9 is not here):", str(error)[:80])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Weights and convolution layers of yolo26n-cls:
- How many times more weights has it than your CNN:

---
# Part G: Make It Your Own and Present (20 min)
Choose **one**:

- Change the number of filters of the first layer (4, 8, 32). How do the accuracy and the number of weights change?
- Remove the pooling layer. What happens to the number of weights of the last layer?
- Add a third convolution layer.
- Train for 5, 15 and 60 epochs and look for overfitting.
- Train the CNN with random shifts of the pictures (augmentation) and measure the shifted test again.

Then give a **3-minute demonstration**: show the digits, the hand-made filter, the two models, the filters and the shift test.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- What a CNN can do that a plain network cannot:

> **Where next.** In Lab 40 you learn to evaluate and tune models in a disciplined way: learning curves, cross-validation and changing one hyperparameter at a time.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The digits are shown and the three sets are made | |
| 2 | The convolution is done by hand and the two ways agree | |
| 3 | The weights of the MLP and the CNN are counted and the shapes of the layers are explained | |
| 4 | Both models are trained and compared, with the time on the GPU | |
| 5 | The filters and the feature maps are shown and described | |
| 6 | The shift test is done and the result is explained | |
| 7 | The confusion matrix is read | |
| 8 | One improvement is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is a filter (kernel) in a CNN? What does it produce?:
- 2. Why does a CNN need fewer weights than a plain network for the same picture?:
- 3. What do padding and stride do?:
- 4. What does max pooling do? Why use it?:
- 5. How many weights does a convolution with 1 input channel, 8 filters and a 3 x 3 kernel have?:
- 6. Why did the CNN lose less than the MLP when the picture was shifted?:
- 7. Why do we move the model and the data to the GPU?:
- 8. What do the early layers of a CNN find? What do the later layers find?:
- 9. The test accuracy of your CNN is much lower than the training accuracy. Name two things to try.:
- 10. Name two labs of Part 2 that use a CNN.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `CUDA out of memory` | Other notebooks or the language model use the shared 8 GB | Stop the language model server (`labview.llm_stop()`), shut down other kernels |
| `Expected all tensors to be on the same device` | The data and the model are not both on the GPU | Use `.to(device)` for both; run Part B Step 1 again |
| The training is slow on the CPU | `device` is `cpu` | Check `torch.cuda.is_available()`; call your instructor if it is False |
| The accuracy is 10 % | The data was not scaled to 0 to 1, or the labels do not match | Run Part B Step 1 again; the images are divided by 16 |
| The pictures do not appear | The plot cell was not run | Run it again; the picture is under the cell |
| `ModuleNotFoundError: sklearn` | scikit-learn is not in the environment | Do Lab 19 (it installs scikit-learn) or ask your instructor |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(39)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Stanford CS231n, Convolutional Networks: https://cs231n.github.io/convolutional-networks/
- PyTorch, Conv2d: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- CNN Explainer (interactive): https://poloclub.github.io/cnn-explainer/